# HeatUp 03 - analysis (four gates)

Reads what is on disk and judges it. **No calculator, no calculation** - results computed by notebook 02, by DFT, or
by anything else that writes the same files all work (see the end of this notebook).

| gate | question | evidence |
|---|---|---|
| **1** mechanical | positive-definite C (Born-Huang)? | relaxed-ion elastic tensor |
| **0** structural transition | did the structure stay in its basin? | block-averaged positions / cell / symmetry breaking from the MD |
| **2** vibrational | soft modes? | MD VDOS (momenta VACF) and harmonic phonons |
| **3** thermodynamic | above the T-dependent hull of its competitors? | G(T) with the *same* method for every phase |

Gates are **independent**: the gates asked for in notebook 01 are reported (override with `cfg.gates.enabled`), every one is
evaluated whatever the others say. Verdicts are always recomputed from cached results, so a threshold never needs a rerun. MD temperatures and
the operating temperature come from each phase's `plan.json` and from the MD runs that exist - nothing to type here.

In [ ]:
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import heatup
from heatup import cfg, override, status, run_analysis, evaluate, evaluate_all
from heatup.analyze.report import (gate_table, style_status, show_status, status_table, show_stability_figure,
                                   plot_free_energy_terms, plot_vdos, plot_fc3, print_layout)

warnings.filterwarnings("ignore")
heatup.verbose()
if os.path.exists("heatup_project.json"):
    heatup.load_json("heatup_project.json")      # folder roots written by notebook 01
print("HeatUp", heatup.__version__, "|", cfg.phases.database_root)

## Configuration

In [ ]:
# -- how trajectories are read (re-runs only the cheap analysis steps) ----------------------------------
cfg.analysis.equil_ps = 5.0          # discarded before analysis (ps)
cfg.analysis.stride = 1
cfg.analysis.block_ps = 2.0          # first/last averaging window of Gate 0 (ps)
cfg.analysis.vdos_max_lag_ps = 10.0  # longest VACF lag of the VDOS (ps)
cfg.analysis.mobile_species = []     # e.g. ["Li"] for a superionic conductor (excluded from Gates 0 and 2)

# -- which gates, and their thresholds (never trigger a recomputation) ----------------------------------
cfg.gates.enabled = None  # None = the gates asked for in notebook 01 (cfg.plan.gates); or e.g. ["mechanical", "thermodynamic"]
cfg.gates.hull_ok_eV = 0.010
cfg.gates.hull_fail_eV = 0.100

# -- free energy and hull ---------------------------------------------------------------------------------
cfg.hull.phonon_method = None  # None = as prepared in notebook 01; or "HA" | "QHA" | "VDOS" to use other data that exist
cfg.hull.target_overlay = "VDOS"  # anharmonic estimate reported next to the consistent hull
cfg.hull.temperatures = list(range(0, 1501, 50))   # T grid of the hull (QHA: inside cfg.qha.temperatures)

# -- optional free-energy contributions (heatup.analyze.ensembles): each reads ONE file under <phase>/disorder/ ----
# A phase without the file contributes 0.  Check what every phase has with ensembles.contribution_inventory(phase).
cfg.hull.include_configurational = True   # configurational block on/off (False = no configurational term at all)
cfg.hull.configurational_model = "ideal"  # "ideal" = -TS of ideal mixing (site_occupancies.json) | "enumerated" = Boltzmann
                                          # ensemble of enumerated configurations (configurations.json) | "monte_carlo" =
                                          # lattice-model MC free energy (monte_carlo.json).  One model at a time.
cfg.hull.include_orientational = False    # hindered rotors / discrete cation orientations (rotors.json)
cfg.hull.include_defects = False          # composition-conserving defect pairs, dilute limit (defects.json)
# operating T and the 0 K pruning window were fixed per phase in notebook 01 (plan.json)

TARGETS = None                           # None = every phase in database_root; or e.g. ["Cu3Au/L1_2"]

## Is everything there?
One table for the whole project: `done` is usable evidence; anything else is listed with the reason. Unfinished or failed
steps are also flagged `[incomplete]` in the verdict of every gate that reads them (`strict=True` makes such a gate `missing`).

In [ ]:
show_status()
display(status_table(status(), problems_only=True))

## Analyse the MD runs and assess the gates
`run_analysis` runs the calculator-free analysis steps (`mdstats`, `vdos`, `transition`) of every finished MD run; cached by manifest.

In [ ]:
run_analysis()
dirs = [os.path.join(cfg.phases.database_root, t) for t in (TARGETS or [])] or None
reports = {f"{r['material']}/{r['label']}": r for r in
           ([evaluate(d) for d in dirs] if dirs else evaluate_all())}
display(style_status(gate_table(reports)))
for t, r in reports.items():
    display(show_stability_figure(r))
    for f in r["flags"]:
        print(t, f)

## Hull in detail: consistent hull vs anharmonic overlay
The consistent curve compares like with like (every phase with the same method). The overlay adds
`F_VDOS - F_HA` of the target only: an *estimate* of the anharmonic stabilisation, not a hull.

In [ ]:
for t, r in reports.items():
    plot_free_energy_terms(r)
    plot_vdos(r)
    plt.show()
    print("decomposition at operating T:", (r["thermodynamic"] or {}).get("decomposition"))

## A subset of gates / strict evidence
Only the listed gates are reported (the others show as `skipped`). `strict=True` turns a gate whose evidence is incomplete into `missing`.

In [ ]:
subset = {t: evaluate(os.path.join(cfg.phases.database_root, t), gates=["mechanical", "thermodynamic"],
                      save_figure=False, strict=True) for t in reports}
for t, r in subset.items():
    print(t, "| enabled:", r["enabled"], "| skipped:", r["skipped"], "| overall:", r["overall"], "| incomplete:", r["incomplete"])

## Anharmonic diagnostics (third-order force constants)
Only if notebook 01/02 planned the `fc3` step (needs `phono3py`): kappa_RTA(T) and phonon linewidths. A diagnostic, never a gate.

In [ ]:
for t, r in reports.items():
    if r.get("anharmonic"):
        plot_fc3(r); plt.show()
        print(t, r["anharmonic"])
    else:
        print(t, "no fc3 result")

## Thresholds can change without recomputing

In [ ]:
with override(**{"gates.hull_ok_eV": 0.002, "gates.zero_frac_warn": 0.002, "gates.transition_rms_A": 0.2}):
    strict = {t: evaluate(os.path.join(cfg.phases.database_root, t), save_figure=False) for t in reports}
display(style_status(gate_table(strict)))

## Results from another code (DFT, another potential)
Notebook 03 only reads files. Put them in the folders notebook 01 created, in the layout of `heatup.core.layout`:

| evidence | file(s) | needed fields |
|---|---|---|
| relaxed structure + energy | `relaxation/CONTCAR`, `relaxation/energy.json` | `converged`, `energy_eV_per_atom`, `enthalpy_eV_per_atom`, `calculator` (a label: phases of one hull must share it) |
| Gate 1 | `elastic/elastic.json` | `C_GPa` (6x6, Voigt), `moduli` (`bulk_hill_GPa`, `shear_hill_GPa`, ...) |
| harmonic phonons, Gates 2 and 3 | `phonons/ha.json` | `modes` (a `ModeSet` dict: `omega_meV`, `weight`), `imag_fraction`, `max_imag_meV` |
| MD, Gates 0 and 2 | `md/<T>K/output.traj` (ASE, with momenta) + `md/<T>K/md.json` | `temperature_K`, `frame_dt_fs` (written last: marks the run complete) |

Files without a HeatUp manifest are reported `external` in the status table and accepted as they are.